In [ ]:
import numpy as np
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split

from students.miller.lesson3 import Exercise

In [ ]:
def normalize_data(X_train, X_val, X_test):

    mean = np.mean(X_train, axis=0)
    std = np.std(X_train, axis=0)

    eps = 1e-8
    std = np.where(std < eps, 1.0, std)

    X_train_norm = (X_train - mean) / std
    X_val_norm = (X_val - mean) / std
    X_test_norm = (X_test - mean) / std

    return X_train_norm, X_val_norm, X_test_norm, mean, std

In [ ]:
class Training:
    def __init__(self, model, loss, learning_rate: float = 0.01):
        self.model = model
        self.loss = loss
        self.learning_rate = learning_rate

    def train_step(self, X_batch: np.ndarray, y_batch: np.ndarray) -> float:
        output = self.model.forward(X_batch)
        loss_value = self.loss.forward(output, y_batch)
        grad = self.loss.backward()
        self.model.backward(grad)

        for param, grad_param in zip(self.model.parameters, self.model.grad, strict=False):
            param -= self.learning_rate * grad_param

        return loss_value

    def evaluate(self, X: np.ndarray, y: np.ndarray) -> tuple:
        output = self.model.forward(X)
        loss_value = self.loss.forward(output, y)
        predictions = np.argmax(output, axis=1)
        accuracy = np.mean(predictions == y)
        return loss_value, accuracy

    def train(
        self,
        X_train: np.ndarray,
        y_train: np.ndarray,
        X_val: np.ndarray,
        y_val: np.ndarray,
        batch_size: int = 32,
        epochs: int = 100,
        verbose: bool = True,
    ):
        n_samples = X_train.shape[0]

        for epoch in range(epochs):
            indices = np.random.permutation(n_samples)
            X_shuffled = X_train[indices]
            y_shuffled = y_train[indices]

            epoch_loss = 0
            n_batches = 0

            for i in range(0, n_samples, batch_size):
                X_batch = X_shuffled[i : i + batch_size]
                y_batch = y_shuffled[i : i + batch_size]
                batch_loss = self.train_step(X_batch, y_batch)
                epoch_loss += batch_loss
                n_batches += 1

            avg_train_loss = epoch_loss / n_batches
            val_loss, val_acc = self.evaluate(X_val, y_val)

            if verbose and (epoch + 1) % 10 == 0:
                print(
                    f"Epoch {epoch + 1}/{epochs} - "
                    f"Train Loss: {avg_train_loss:.4f}, "
                    f"Val Loss: {val_loss:.4f}, "
                    f"Val Acc: {val_acc:.4f}"
                )


def create_digits_model(input_size: int = 64, hidden_sizes: list[int] | None = None, output_size: int = 10):
    if hidden_sizes is None:
        hidden_sizes = [128, 64]

    layers = []
    layers.append(Exercise.create_linear_layer(input_size, hidden_sizes[0]))
    layers.append(Exercise.create_relu_layer())

    for i in range(len(hidden_sizes) - 1):
        layers.append(Exercise.create_linear_layer(hidden_sizes[i], hidden_sizes[i + 1]))
        layers.append(Exercise.create_relu_layer())

    layers.append(Exercise.create_linear_layer(hidden_sizes[-1], output_size))
    layers.append(Exercise.create_logsoftmax_layer())

    return Exercise.create_model(*layers)


def main():
    digits = load_digits()
    X = digits.data.astype(np.float32)
    y = digits.target

    print(f"Размер данных: {X.shape}")
    print(f"Количество классов: {len(np.unique(y))}")

    X_temp, X_test, y_temp, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
    X_train, X_val, y_train, y_val = train_test_split(X_temp, y_temp, test_size=0.25, random_state=42, stratify=y_temp)

    print("\nРазмер выборок:")
    print(f"  Train: {X_train.shape[0]}")
    print(f"  Val: {X_val.shape[0]}")
    print(f"  Test: {X_test.shape[0]}")

    X_train_norm, X_val_norm, X_test_norm, mean, std = normalize_data(X_train, X_val, X_test)

    model = create_digits_model(input_size=64, hidden_sizes=[128, 64], output_size=10)
    loss = Exercise.create_nll_loss()
    training = Training(model, loss, learning_rate=0.01)

    training.train(X_train_norm, y_train, X_val_norm, y_val, batch_size=32, epochs=100, verbose=True)

    test_loss, test_acc = training.evaluate(X_test_norm, y_test)
    print(f"Тестовые потери: {test_loss:.4f}")
    print(f"Тестовая точность: {test_acc:.4f}")

    print("Статистика по каждому классу:")

    output = model.forward(X_test_norm)
    predictions = np.argmax(output, axis=1)

    for digit in range(10):
        mask = y_test == digit
        if np.sum(mask) > 0:
            acc = np.mean(predictions[mask] == digit)
            print(f"Цифра {digit}: {acc:.4f} ({np.sum(mask)} примеров)")

    return model, training, test_acc


if __name__ == "__main__":
    model, training, accuracy = main()